# LLM Gateway Concepts Using LiteLLM library.

In [1]:
import warnings 
from litellm import completion
warnings.filterwarnings("ignore")

In [2]:
import os
import dotenv
dotenv.load_dotenv()

print("OpenAI API key loaded:     ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Gemini API key loaded:     ", "✅" if os.getenv("GOOGLE_API_KEY") else "❌")
print("Groq API key loaded:     ", "✅" if os.getenv("GROQ_API_KEY") else "❌")
print("Anthropic API key loaded:     ", "✅" if os.getenv("ANTHROPIC_API_KEY") else "❌")

OpenAI API key loaded:      ✅
Gemini API key loaded:      ✅
Groq API key loaded:      ✅
Anthropic API key loaded:      ❌


## Simplest LiteLLM Example

In [3]:
response_openAI = completion(
    model="gpt-4o-mini",
    messages=[{'role':'user','content':'explain RAG in two to three sentences'}]
)
print("🔵 OpenAI: ", response_openAI.choices[0].message.content) #type:ignore

response_groq = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{'role':'user','content':'explain RAG in AI in two to three sentences'}]
)
print("🟢 Groq: ", response_groq.choices[0].message.content) #type:ignore

🔵 OpenAI:  RAG, or Retrieval-Augmented Generation, is a framework that enhances language models by integrating retrieval capabilities, allowing them to access external documents or knowledge sources during the generation process. This hybrid approach enables the model to produce more accurate and contextually relevant responses by leveraging up-to-date information beyond its training data. RAG is particularly useful for tasks requiring extensive factual knowledge and real-time data.
🟢 Groq:  RAG stands for "Retrieve, Assess, Generate," which is a key concept in the field of Artificial Intelligence (AI), specifically in applications like question-answering models and chatbots. It involves the AI system Retrieving relevant information from a database or knowledge base, Assessing the relevance and accuracy of the information, and finally Generating a response based on the retrieved and assessed information. This approach helps AI systems provide accurate and informed responses to user que

In [4]:
import litellm
import logging

In [5]:
prompt = "Eplain RAG in AI in one sentence."

providers = [
    ("🔵 OpenAI", 'gpt-4o-mini'),
    ("🟢 Groq", 'groq/llama-3.1-8b-instant'),
    ("🟡 Gemini", 'gemini/gemini-2.5-flash'),
    ("🟣 Anthropic", 'claude-3-5-haiku-20241022')
]

for label,model in providers:
    try:
        res = completion(model=model, messages=[{'role':'user','content':prompt}])
        print(f'{label:<15}: {res.choices[0].message.content}')#type:ignore
    except Exception as e:
        print(f'{label:<15}: {type(e).__name__}')

warnings.filterwarnings('ignore')

🔵 OpenAI       : RAG, or Retrieval-Augmented Generation, is an AI approach that combines the generation capabilities of language models with external information retrieval, allowing the model to access and incorporate relevant data from a knowledge base to enhance its responses.
🟢 Groq         : RAG in AI stands for Retrieval-Augmented Generation, a technique in natural language processing where a large language model generates text based on information retrieved from a separate large knowledge base or database.
🟡 Gemini       : RAG (Retrieval-Augmented Generation) improves AI answers by first retrieving relevant external information and then using it to generate a more accurate, current, and grounded response.

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

🟣 Anthropic    : BadRequestError


## Automatic Fallbacks - When a Primary LLM provider Goes Down
**Read Story:** OpenAI had a 4hr outage in November 2023. All the apps that used or Hard-coded gpt-4 went completly dark. Check the [offical Blog.](https://blog.scottlogic.com/2023/11/16/OpenAI-Outage-November-2023.html) 

With a gateway, if one provider fails, we automatically fallbacks to the another. Production apps must have this.

In [ ]:

res = completion(
    model='claude-3-5-haiku-20241022',
    messages=[{'role':'user',"content":"What is an LLM gateway"}],
    fallbacks=['gpt-4o-mini','groq/llama-3.1-8b-instant']
)

print("Response :" , res.choices[0].message.content[:200], "....") #type:ignore
print("Which model actually answered -> ", res.model)

20:47:11 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model claude-3-5-haiku-20241022: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=claude-3-5-haiku-20241022
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
Traceback (most recent call last):
  File "d:\Lang_Graph_Concepts\langgraphvenv\Lib\site-packages\litellm\litellm_core_utils\fallback_utils.py", line 62, in async_completion_with_fallbacks
    response = await litellm.acompletion(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Lang_Graph_Concepts\langgraphvenv\Lib\site-packages\litellm\utils.py", line 1871, in wrapper_async
    raise e
  File "d:\Lang_Graph_Concepts\langgraphvenv\Lib\site-packages\litellm\utils.py", line 1690, in wrapper_async
    result = await original_function(*args, **kwargs)
             ^^^^^^^^^^


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



Task was destroyed but it is pending!
task: <Task pending name='Task-173' coro=<LoggingWorker._worker_loop() running at d:\Lang_Graph_Concepts\langgraphvenv\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:109>>



Provider List: https://docs.litellm.ai/docs/providers

Response : An LLM gateway typically refers to a system or interface that allows users to interact with a Large Language Model (LLM). These models, such as GPT-3, GPT-4, and others, are designed to understand and ....
Which model actually answered ->  gpt-4o-mini-2024-07-18


In [8]:
res = completion(
    model='Non-existent_model',
    messages=[{'role':'user',"content":"What is an LLM gateway"}],
    fallbacks=['groq/llama-3.1-8b-instant','gpt-4o-mini']
)

print("✅ App still got a response!! Eventhough the Primary llm provider failed")
print("Response :" , res.choices[0].message.content[:200], "....Rest of the content") #type:ignore
print("Which model actually answered -> ", res.model)

20:32:48 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model Non-existent_model: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=Non-existent_model
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
Traceback (most recent call last):
  File "d:\Lang_Graph_Concepts\langgraphvenv\Lib\site-packages\litellm\litellm_core_utils\fallback_utils.py", line 62, in async_completion_with_fallbacks
    response = await litellm.acompletion(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Lang_Graph_Concepts\langgraphvenv\Lib\site-packages\litellm\utils.py", line 1871, in wrapper_async
    raise e
  File "d:\Lang_Graph_Concepts\langgraphvenv\Lib\site-packages\litellm\utils.py", line 1690, in wrapper_async
    result = await original_function(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



Task was destroyed but it is pending!
task: <Task pending name='Task-130' coro=<LoggingWorker._worker_loop() running at d:\Lang_Graph_Concepts\langgraphvenv\Lib\site-packages\litellm\litellm_core_utils\logging_worker.py:109>>



Provider List: https://docs.litellm.ai/docs/providers

✅ App still got a response!! Eventhough the Primary llm provider failed
Response : An LLM (Large Language Model) gateway is a software interface or platform that connects users or applications to large language models (LLMs), allowing them to interact with the model's capabilities t ....Rest of the content
Which model actually answered ->  llama-3.1-8b-instant


## Cost Tracking - Know where your Money goes
LiteLLM **automatically calculates** the cost of evry call using its **built-in pricing database**. No more suprise *Bills*.

In [9]:
from litellm import completion_cost

In [12]:
res = completion(
    model='groq/llama-3.1-8b-instant',
    messages=[{'role':'user',"content":"Write a Haiku on AI"}]
)

cost = completion_cost(completion_response=res,model='groq/llama-3.1-8b-instant')

print("Response :" , res.choices[0].message.content) #type:ignore
print("\n Input Tokens -> ", res.usage.prompt_tokens)#type:ignore
print("\n Output Tokens -> ", res.usage.completion_tokens)#type:ignore
print(f"Total Cost:      ${cost:.8f}")

Response : Code and circuit mind
Learning, adapting, it grows
Future's silent grasp

 Input Tokens ->  41

 Output Tokens ->  17
Total Cost:      $0.00000341
